# Business Understanding

**Potential Stakeholders**: WNBA Management, Team Executives, Sports Analysts and Commentators, Agents, Fans

**Stakeholder Actions**:
- **WNBA Management**: Use predictions to inform marketing strategies and policy decisions.
- **Team Executives**: Apply insights to guide roster moves, coaching hires, and playoff preparation.
- **Sports Analysts and Commentators**: Leverage forecasts to enrich game coverage and provide deeper analysis to audiences.
- **Agents**: Identify emerging talent and award contenders for targeted representation and contract negotiations.
- **Fans**: Engage with predictive content to enhance season experience and participate in discussions around team and player performance.

**Available Data**: 10 years of comprehensive WNBA data including player statistics, team performance, coaching records, and awards history. The main risks involve missing values, data inconsistencies, limited sample size, potential class imbalance and integrating the datasets, all of which may affect model accuracy and reliability.

**Business Objectives**:
1. **Improve Team Performance Prediction**: Develop models to accurately predict conference rankings
   - Success Metric: Correctly rank teams within ±2 positions of actual final standings
   - Business Value: Enables better strategic planning for playoffs and roster management

2. **Optimize Coaching Decisions**: Predict which teams are likely to change coaches
   - Success Metric: Identify 70%+ of teams that will undergo coaching changes
   - Business Value: Reduces costs of poor coaching fits and improves team stability

3. **Identify Award Winners Early**: Predict individual award recipients
   - Success Metric: Correctly predict 60%+ of major award winners
   - Business Value: Enhances marketing campaigns and fan engagement throughout season; allows agents and teams to identify promising players to work with

**Secondary Goals**:
- Understand key performance indicators that differentiate successful teams
- Identify player attributes that correlate with award recognition
- Analyze impact of coaching experience on team performance

## Business Goals

### Goal 1: Conference Ranking Prediction
**Data Mining Task**: Regression/Ranking Problem
- **Target Variable**: Team rank within conference
- **Features**: 
  - Team statistics: offensive/defensive efficiency, win%, home/away performance
  - Player statistics: NBA_PER, assist-to-turnover ratio, shooting percentages
  - Historical performance: previous season rankings, playoff appearances
- **Approach**: Ordinal regression or learning-to-rank algorithms
- **Evaluation Metric**: Mean Absolute Error (MAE) in ranking position, Spearman correlation

### Goal 2: Coaching Change Prediction
**Data Mining Task**: Binary Classification Problem
- **Target Variable**: Coach change (Yes/No) in upcoming season
- **Features**:
  - Team performance: win%, playoff participation, win_pct variance
  - Coach statistics: experience (seasons coached), historical win%, recent performance trend
  - Situational factors: consecutive losing seasons, missing playoffs
- **Approach**: Logistic Regression, Random Forest, XGBoost
- **Evaluation Metric**: Precision, Recall, F1-Score (class imbalance likely)

### Goal 3: Award Winner Prediction
**Data Mining Task**: Multi-class Classification Problem (per award category)
- **Target Variables**: 
  - MVP
  - Rookie of the Year
  - Defensive Player of the Year
  - Most Improved Player
- **Features**:
  - Player statistics: NBA_PER, ppg, apg, rpg, spg, bpg, True Shooting %
  - Team context: team win%, playoff success
  - Player position and role
  - Historical award patterns by position
- **Approach**: Multi-class classification (Random Forest, Neural Networks)
- **Evaluation Metric**: Top-3 Accuracy, Precision@K

## Project Plan

**Phase 1: Data Understanding & Preparation**
- Explore and assess data quality across all datasets (teams, players, coaches, awards)
- Handle missing values, outliers, and inconsistencies
- Engineer relevant features for each prediction task
- Create unified dataset with proper temporal alignment

**Phase 2: Model Development**
- Conference Ranking Prediction models
- Coaching Change Prediction models  
- Award Winner Prediction models
- Establish baseline models and iterate with parameter tuning

**Phase 3: Model Evaluation & Selection**
- Compare models against success metrics
- Perform cross-validation and temporal validation
- Select best-performing models for each task

# Data Understanding

## Setup & Data Collection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [ ]:
sns.set_theme(style="whitegrid", palette="pastel")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
awards_players_df = pd.read_csv('../datasets/awards_players.csv')
coaches_df        = pd.read_csv('../datasets/coaches.csv')
players_teams_df  = pd.read_csv('../datasets/players_teams.csv')
players_df        = pd.read_csv('../datasets/players.csv')
series_post_df    = pd.read_csv('../datasets/series_post.csv')
teams_post_df     = pd.read_csv('../datasets/teams_post.csv')
teams_df          = pd.read_csv('../datasets/teams.csv')

## Data Description

### *awards_players*

In [ ]:
display(awards_players_df.describe(include='all'))
print("\nMissing Values per Column:")
display(awards_players_df.isnull().sum())
print(f"Number of duplicate rows: {awards_players_df.duplicated().sum()}")

### *coaches*

In [ ]:
display(coaches_df.describe(include='all'))
print("\nMissing Values per Column:")
display(coaches_df.isnull().sum())
print(f"Number of duplicate rows: {coaches_df.duplicated().sum()}")

### *players_teams*

In [ ]:
display(players_teams_df.describe(include='all'))
print("\nMissing Values per Column:")
display(players_teams_df.isnull().sum())
print(f"Number of duplicate rows: {players_teams_df.duplicated().sum()}")

### *players*

In [ ]:
display(players_df.describe(include='all'))
print("\nMissing Values per Column:")
display(players_df.isnull().sum())
print(f"Number of duplicate rows: {players_df.duplicated().sum()}")

### *series_post*

In [ ]:
display(series_post_df.describe(include='all'))
print("\nMissing Values per Column:")
display(series_post_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {series_post_df.duplicated().sum()}")

### *teams_post*

In [ ]:
display(teams_post_df.describe(include='all'))
print("\nMissing Values per Column:")
display(teams_post_df.isnull().sum())
print(f"Number of duplicate rows: {teams_post_df.duplicated().sum()}")

### *teams*

In [ ]:
display(teams_df.describe(include='all'))
print("\nMissing Values per Column:")
display(teams_df.isnull().sum())
print(f"Number of duplicate rows: {teams_df.duplicated().sum()}")

### Data Description Report

#### Redundant Columns

Across almost all datasets, there are columns containing only a single unique value. These provide no predictive power and should be dropped to reduce dimensionality.

- **League Identifiers**: The lgID column (and variations like lgIDWinner, lgIDLoser) appears in all datasets with the single value "WNBA".

- **Zero-Variance Metrics**: In the *teams* dataset, the following columns contain only zeros (0):

    - seeded

    - tmORB, tmDRB, tmTRB

    - opptmORB, opptmDRB, opptmTRB

    The same happens with these columns from *players*:

    - firstseason, lastseason

#### Missing or Null Values

The *teams* and *players* datasets are affected by missing or null values.

1. *teams*

    - Missing values in firstRound, semis, and finals are not errors but indicators that a team did not advance to those playoff stages.

    - The divID column is entirely empty and provides no value. Therefore, it should be removed.

2. *players*

    - The pos attribute is missing for 78 players.

    - The college and collegeOther columns are missing for a significant number of players (167 and 882 out of 893, respectively). The latter is essentially unusable.

    - There are values of 0 in the weight and height columns, which should be impossible.

    - The most frequent value for birthDate is "0000-00-00" (appearing 84 times), which clearly indicates missing data rather than valid dates. The deathDate column is predominantly "0000-00-00", which is expected for living players.